# Friday: trustworthy measurements, and what the system knows

*PhDOpen MIMUW, 16 October 2026. SeBS tutorial, part 2 of 2.*

**Take-away:** repeat, control, and look underneath: the numbers only make sense once you know what produced them.

Today you'll work with SeBS's **local** backend: each function runs in a Docker container on your laptop. That's where you control the hardware: which CPU runs the function, and who else runs there.

## 1. Rerun Thursday in the cloud

▶️ Setup check. Your OpenWhisk cluster isn't needed today: if your laptop is short on memory, stop it (`openwhisk/stop.sh`).

In [ ]:
import sebs_tutorial as st
st.check("friday")

▶️ The same cloud deployment as on Thursday (set `CLOUD` to the cloud you used). The function is still deployed: SeBS finds it in its cache (if you didn't deploy it on Thursday, SeBS does now).

In [ ]:
import time
from sebs import SeBS
from sebs.faas.function import Trigger

CLOUD = "aws"   # or "gcp": the same cloud as on Thursday

sebs_client = SeBS(cache_dir="cache", output_dir="outputs", verbose=False, logging_filename=None)
cloud_config = st.cloud_config(CLOUD)
cloud = sebs_client.get_deployment(cloud_config)
cloud.initialize(resource_prefix=st.cloud_prefix())
cloud_experiment = sebs_client.get_experiment_config(cloud_config["experiments"])

thumb_cloud = sebs_client.get_benchmark("210.thumbnailer", cloud, cloud_experiment)
thumb_cloud_input = thumb_cloud.prepare_input(cloud.system_resources, size="small")
fn_cloud = cloud.get_function(thumb_cloud)
cloud_trigger = st.http_trigger(cloud, fn_cloud)

print(f"function {fn_cloud.name}, {fn_cloud.config.memory} MB")

▶️ **Thursday's data first:** your 30 invocations from yesterday (`results/thursday_<cloud>.csv`, section 4 of Thursday's notebook), and the instructor's run of the same experiment (`reference/`: `210.thumbnailer`, `small` input, default memory, AWS `eu-central-1`, 15 October). If you didn't get to Thursday's measurement, compare with the instructor's.

In [ ]:
thursday = {}
for name, path in [("Thursday (yours)", f"results/thursday_{CLOUD}.csv"),
                   ("Thursday (instructor)", f"reference/thursday_{CLOUD}.csv")]:
    if (st.TUTORIAL_DIR / path).exists():
        thursday[name] = st.load(path)
        print(f"{name}: {len(thursday[name])} invocations from {path}")
    else:
        print(f"{name}: no {path}")

st.compare_ci(thursday)

✏️ **TODO 1:** 30 warm invocations, exactly as on Thursday. Thursday's run used the function's default memory; the first lines set it back after the sweep (`update_function_configuration` changes the cloud, not SeBS's cache, so `fn_cloud.config.memory` can't tell you what the cloud runs).

In [ ]:
# The cloud may still run with the last size of Thursday's sweep: SeBS's cache doesn't know.
fn_cloud.config.memory = thumb_cloud.benchmark_config.memory
cloud.update_function_configuration(fn_cloud, thumb_cloud)

fri_results = ...   # TODO: 30 invocations of cloud_trigger with thumb_cloud_input
fri = st.to_frame(fri_results)

▶️ Median and bootstrap 95% confidence interval of every component, both days:

In [ ]:
st.compare_ci({**thursday, "Friday": fri})

✅

In [ ]:
print("✔ 30 invocations today" if len(fri) == 30 else f"✘ {len(fri)} invocations today, not 30")

❓ Do you see any major difference between the days? If you compare with the instructor's run: which components can you compare, and which not?

## 2. Interference: add CPU pinning to SeBS, then measure

To control interference, you must control *where* things run. SeBS's local backend can't pin containers to CPUs yet. In `sebs/local/local.py`, in `_start_container`, there's a `# FIXME: make CPUs configurable`. You'll close that FIXME.

### Step 1: implement pinning

▶️ Open the file in JupyterLab (the link opens the editor):

In [ ]:
st.sebs_source("sebs/local/local.py")

✏️ **TODO 2 (edit SeBS, in the editor):** in `_start_container`, just before `container = self._docker_client.containers.run(**container_kwargs)`:

```python
# TODO: if the environment variable SEBS_LOCAL_CPUSET is set (e.g. "2"),
#       make Docker run the container only on those CPUs
```

The notebook sets `SEBS_LOCAL_CPUSET` for you, and SeBS runs in this kernel, so `os.environ.get("SEBS_LOCAL_CPUSET")` in SeBS sees it. Your job is to find out how Docker should be configured. SeBS starts containers with the Docker SDK for Python: every key of `container_kwargs` is an argument of [`containers.run`](https://docker-py.readthedocs.io/en/stable/containers.html). Which argument restricts a container to a set of CPUs, and in what format does it take them? Docker's guide to [resource constraints](https://docs.docker.com/engine/containers/resource_constraints/) explains the `docker run` flags behind these arguments. Careful: several CPU options limit *how much* CPU a container gets, not *which* CPUs it runs on. Pinning applies when a container starts.

Save the file, then **restart the kernel** (Kernel → Restart Kernel): Python loads SeBS's code only once. Then run the setup cell below.

🚀 **Level 2:** make it a configuration option (`deployment.local.cpuset`, read in `sebs/local/config.py`), or close the neighboring FIXMEs: `nano_cpus` for a CPU quota, `mem_limit` from the function's memory.

▶️ **Setup for the local backend.** Rerun this cell after every kernel restart.

In [ ]:
import os, time
import sebs_tutorial as st
from sebs import SeBS
from sebs.faas.function import Trigger

sebs_client = SeBS(cache_dir="cache", output_dir="outputs", verbose=False, logging_filename=None)
local_config = st.load_config("local")
local_config["deployment"]["local"]["storage"] = st.start_storage()
local = sebs_client.get_deployment(local_config)
local.initialize()
experiment = sebs_client.get_experiment_config(local_config["experiments"])


def http_trigger(func):
    # Local functions have no trigger until we create one.
    return (func.triggers(Trigger.TriggerType.HTTP) or [local.create_trigger(func, Trigger.TriggerType.HTTP)])[0]


def invoke(trigger, payload, n):
    return st.to_frame([trigger.sync_invoke(payload) for _ in range(n)])

### Step 2: choose the CPUs

For the experiment, you need three CPUs: one for the function, and two places for a noisy neighbor: the other hardware thread of the function's core (its *SMT sibling*, "hyper-threading"), and a CPU on another physical core. Which CPU is which depends on your processor.

The noisy neighbor is [stress-ng](https://github.com/ColinIanKing/stress-ng), a load generator, in its own container. ▶️ Build its image: Alpine Linux (a 3.5 MB base image) plus the `stress-ng` package. Docker downloads both once; if `./check_setup.sh friday` already built the image, every step says `Using cache`.

In [ ]:
import io
import docker

STRESS_DOCKERFILE = '''
FROM alpine:3.20
RUN apk add --no-cache stress-ng
ENTRYPOINT ["stress-ng"]
'''
image, build_log = docker.from_env().images.build(
    fileobj=io.BytesIO(STRESS_DOCKERFILE.encode()), tag=st.STRESS_IMAGE, rm=True
)
for entry in build_log:
    print(entry.get("stream", ""), end="")
print(f"{st.STRESS_IMAGE}: {image.attrs['Size'] / 1e6:.0f} MB")

▶️ How Docker sees your CPUs (on macOS and Windows: the CPUs of Docker Desktop's VM). The same image runs a small shell script that reads `/sys/devices/system/cpu`:

In [ ]:
topology = st.cpu_topology()
topology

- `cpu`: the number Linux and Docker use for the CPU (a hardware thread);
- `core`, `socket`: the physical core and the processor package it belongs to; CPUs with the same `core` (and `socket`) share that core's execution units and its L1 and L2 caches;
- `smt_siblings`: all CPUs of that core; one CPU per core means no SMT;
- `max_mhz`: the highest clock of the CPU, if Linux reports it;
- `type` (Intel's hybrid CPUs only): `P` for performance cores, `E` for efficiency cores.

✏️ **TODO 3:** choose the three CPUs. Write them as strings (Docker's format, e.g. `"2"`):
- **function:** any CPU but CPU 0, which handles many interrupts and system tasks;
- **smt_sibling:** the other CPU of the function's core, or `None` if your CPU has no SMT;
- **other_core:** a CPU on a different physical core, of the same kind as the function's (the same `type` and `max_mhz`): otherwise, you compare different hardware, not interference.

In [ ]:
CPUS = {
    "function": ...,      # TODO
    "smt_sibling": ...,   # TODO: or None
    "other_core": ...,    # TODO
}

✅

In [ ]:
st.check_cpus(CPUS)

### Step 3: verify

▶️ Deploy `110.dynamic-html` locally, pinned to the function's CPU. `enforce_cold_start` replaces the function's container, and the new container starts with the current settings. We use the `large` input: rendering a page of 100,000 random numbers takes about 0.1 s of CPU time.

In [ ]:
os.environ["SEBS_LOCAL_CPUSET"] = CPUS["function"]
html = sebs_client.get_benchmark("110.dynamic-html", local, experiment)
html_input = html.prepare_input(local.system_resources, size="large")
fn_loc = local.get_function(html)
local.enforce_cold_start([fn_loc], html)
trigger = http_trigger(fn_loc)

✅ Docker's configuration and the container's cgroup must both say the function's CPU:

In [ ]:
st.check_pinning(fn_loc, expected=CPUS["function"])

### Step 4: measure under interference

| Condition | Function | Stressor (`stress-ng --cpu 1`) |
|---|---|---|
| A: alone | function CPU | none |
| B: same CPU | function CPU | on the function's CPU |
| C: other core | function CPU | on a CPU of another physical core |

`st.stressor(cpus=...)` starts a pinned stress container when the `with` block begins and stops it at the end. `invoke(trigger, payload, n)` (from the setup cell) returns a DataFrame of `n` invocations.

✏️ **TODO 4:**

In [ ]:
invoke(trigger, html_input, 5)   # warm-up
A = invoke(trigger, html_input, 30)
with st.stressor(cpus=CPUS["function"]):
    B = ...   # TODO: 30 invocations
with st.stressor(cpus=CPUS["other_core"]):
    C = ...   # TODO: 30 invocations

In [ ]:
invoke(trigger, html_input, 5)   # warm-up
A = invoke(trigger, html_input, 30)
with st.stressor(cpus=CPUS["function"]):
    B = invoke(trigger, html_input, 30)
with st.stressor(cpus=CPUS["other_core"]):
    C = invoke(trigger, html_input, 30)

▶️ Slowdown of the median client time, with a bootstrap 95% CI:

In [ ]:
slow = st.slowdown(A, {"same CPU": B, "other core": C})
slow

✅

In [ ]:
if slow.loc["same CPU", "slowdown"] > 1.3:
    print("✔ the stressor on the same CPU slows the function down")
else:
    print("✘ no clear slowdown: is your stressor running on the same CPU? Check st.check_pinning(fn_loc, CPUS['function'])")

▶️ **One more condition.** Most laptop CPUs run two hardware threads per physical core (SMT, "hyper-threading"). They share the core's execution units and caches. What happens with a stressor on the SMT sibling of the function's CPU?

In [ ]:
if CPUS["smt_sibling"] is None:
    print("Your CPU has no SMT siblings (or Docker doesn't show them).")
else:
    with st.stressor(cpus=CPUS["smt_sibling"]):
        S = invoke(trigger, html_input, 30)
    display(st.slowdown(A, {"same CPU": B, "SMT sibling": S, "other core": C}))

❓ With the stressor on the same CPU, why is the slowdown close to 2? Why is `other core` not exactly 1.0 on some laptops, and why is the SMT sibling somewhere in between?

🚀 **Going further:**
- Repeat A and B a few times with the `test` input (about 1.5 ms per invocation).
- `st.stressor(cpus=CPUS["other_core"], kind="stream")` runs a memory-bandwidth stressor on another core. Does pinning protect the function from it? Try it with the `memory` workload of the next section.

## 3. What does the system know about one invocation?

The benchmark `050.os-counters` runs a small workload and reports what the OS and CPU counted, in two tiers:
- **posix**: `getrusage(RUSAGE_THREAD)`, what the operating system counts for every thread (CPU time, page faults, context switches), always available on Linux. *You'll implement this one.*
- **papi**: the CPU's hardware counters (instructions, cycles, cache misses) through the PAPI library, with SeBS's small binding `sebs_papi`, which comes with SeBS's local images.

When a tier isn't available, the output says why. Workloads: `compute` (a Python loop of `size` iterations), `memory` (allocate `size` MB and write one byte per 4 KiB page), `sleep` (`size` ms).

The benchmark lives in this repository, in `benchmarks/050.os-counters/`: a `config.json` (memory, timeout, languages), an `input.py` (inputs for SeBS's test/small/large sizes), and the function code in `python/function.py`. Any benchmark you write for SeBS has this shape.

### Step 1: write the POSIX tier, deploy, measure

▶️ Make the benchmark visible to SeBS, deploy it unpinned, and run it once. Open `function.py` from the link and read `handler`: every tier has `start()`, `stop()`, and `report()`.

In [ ]:
os.environ.pop("SEBS_LOCAL_CPUSET", None)
st.register_benchmark("050.os-counters")
counters_bench = sebs_client.get_benchmark("050.os-counters", local, experiment)
fn_cnt = local.get_function(counters_bench)
local.enforce_cold_start([fn_cnt], counters_bench)   # a fresh, unpinned container
cnt_trigger = http_trigger(fn_cnt)

r = cnt_trigger.sync_invoke({"workload": "compute", "size": 5_000_000})
st.counter_tiers(r)

✏️ **TODO 5 (edit `function.py`):** the `posix` tier says "not implemented yet". Complete the class `Rusage`:
- `start()`: take a snapshot with `resource.getrusage(resource.RUSAGE_THREAD)` and keep it;
- `stop()`: take another snapshot, and store in `self.events` how much each field in `self.FIELDS` grew, e.g. `{"ru_minflt": 51200, ...}`.

*Hint:* `getattr(usage, "ru_minflt")` reads a field by name. Save the file.

▶️ Redeploy and run the three workloads. SeBS hashes the benchmark's code: `get_function` notices your change, rebuilds the code package, and replaces the container. No kernel restart needed: the code runs in the container, not in this notebook. Rerun this cell after every edit.

In [ ]:
counters_bench = sebs_client.get_benchmark("050.os-counters", local, experiment)
fn_cnt = local.get_function(counters_bench)
cnt_trigger = http_trigger(fn_cnt)

r = cnt_trigger.sync_invoke({"workload": "compute", "size": 5_000_000})
mem = cnt_trigger.sync_invoke({"workload": "memory", "size": 200})   # MB
slp = cnt_trigger.sync_invoke({"workload": "sleep", "size": 200})    # ms
st.counters_table({"compute": r, "memory": mem, "sleep": slp})

✅

In [ ]:
st.check_counters({"compute": r, "memory": mem, "sleep": slp})

🚀 Add a fourth workload to `WORKLOADS` in `function.py`, e.g. `io`: write `size` MB to a file in `/tmp` and call `os.fsync`. Redeploy and compare its counters with `sleep`.

### Step 2: what can your machine count?

▶️

In [ ]:
st.counter_tiers(r)

❓ Is your `papi` status `ok`, `partial`, or `unavailable`? If it's not `ok`, what's the error: "no hardware counters exist here", or "not allowed"? And `PMU exposed` in the machine line: what does it tell you?

❓ Why does SeBS start local containers with `privileged` and `seccomp:unconfined` (see `_start_container`)?

▶️ Compare the hardware counters of the three workloads from Step 1:

In [ ]:
st.counters_table({"compute": r, "memory": mem, "sleep": slp}).filter(like="papi", axis=0)

❓ Which workload has the lower IPC (`papi.IPC`, instructions per cycle), and why? Why does `sleep` count almost nothing?

🚀 **Native Linux only.** The function runs as an ordinary user, so the host's `perf_event_paranoid` decides: with 2, PAPI counts only user space; with 3 or 4 (some distributions' default, e.g. Ubuntu 26.04), nothing at all ("Permission level does not permit operation"). In a terminal:
```bash
cat /proc/sys/kernel/perf_event_paranoid
sudo sysctl -w kernel.perf_event_paranoid=2     # restore the old value afterwards!
```

### Step 3: explain the slowdown from section 2

▶️ Pin the counters benchmark to the function's CPU:

In [ ]:
os.environ["SEBS_LOCAL_CPUSET"] = CPUS["function"]
local.enforce_cold_start([fn_cnt], counters_bench)
st.check_pinning(fn_cnt, expected=CPUS["function"])

✏️ **TODO 6:** the same invocation, alone and under the stressor on the same CPU.

In [ ]:
cnt_trigger.sync_invoke({"workload": "compute", "size": 100_000})   # warm-up: the new container imports the code
alone = cnt_trigger.sync_invoke({"workload": "compute", "size": 5_000_000})
with st.stressor(cpus=CPUS["function"]):
    shared = ...   # TODO: the same invocation under the stressor
st.counters_table({"alone": alone, "same CPU": shared})

✅

In [ ]:
a, s = st.counters(alone), st.counters(shared)
more_preempted = s["posix.ru_nivcsw"] > a["posix.ru_nivcsw"]
waits_longer = s["wall_ms"] / s["thread_cpu_ms"] > a["wall_ms"] / a["thread_cpu_ms"]
print("✔ more involuntary context switches, larger wall/CPU ratio" if more_preempted and waits_longer
      else "✘ no difference: is the function pinned (check above)?")

❓ CPU time and instructions barely changed, but wall time grew. Where did the time go?

### Step 4: can the function see its own cold start?

▶️ `enforce_cold_start` replaces the container; the first invocation then imports the function code again.

In [ ]:
local.enforce_cold_start([fn_cnt], counters_bench)
cold_r = cnt_trigger.sync_invoke({"workload": "compute", "size": 100_000})
warm_r = cnt_trigger.sync_invoke({"workload": "compute", "size": 100_000})
st.counters_table({"cold": cold_r, "warm": warm_r}, with_client_time=True)

✅

In [ ]:
# Two witnesses: the function's own flag, and SeBS's local server (first request of a container)
print("function:", st.counters(cold_r)["cold"], st.counters(warm_r)["cold"], "| SeBS:", cold_r.stats.cold_start, warm_r.stats.cold_start)
ok = st.counters(cold_r)["cold"] and not st.counters(warm_r)["cold"] and cold_r.stats.cold_start and not warm_r.stats.cold_start
print("✔ cold, then warm" if ok else "✘ expected a cold and a warm invocation: rerun the cell")

### Step 5 (if PAPI works on your machine): counters over time (optional)

So far, every counter was one number for the whole invocation. PAPI can also *sample*: every N occurrences of one event (an *overflow*), it records the current values of all events. You get a time series and can see the phases of a function.

The `phases` workload (see `function.py`) computes, allocates a 256 MB buffer (far larger than the caches), reads it at random places, and computes again.

▶️ Is PAPI available? `ok` or `partial` means yes; `unavailable` says why. On Docker Desktop and WSL2, there are usually no hardware counters at all: skip this step then. On native Linux, "Permission level does not permit operation" means `perf_event_paranoid` is above 2: see the 🚀 of Step 2.

In [ ]:
papi_status = st.benchmark_output(r)["result"]["papi"]
print(papi_status["status"], papi_status["errors"])

▶️ Sample instructions, cycles, and L1 data-cache misses every 2 million instructions.

In [ ]:
papi_config = {
    "events": ["PAPI_TOT_INS", "PAPI_TOT_CYC", "PAPI_L1_DCM"],
    "overflow_event": "PAPI_TOT_INS",
    "threshold": 2_000_000,
}
ph = cnt_trigger.sync_invoke({"workload": "phases", "size": 2_000_000, "papi": papi_config})
samples = st.papi_samples(ph)   # cumulative counts at each sample
samples.head()

✏️ **TODO 7:** the counters are cumulative. Compute, for each interval between two samples, the IPC (instructions per cycle) and the L1 data-cache misses per 1,000 instructions, and plot both over time.

*Hint:* `samples.diff()` gives the counts between consecutive samples.

In [ ]:
import matplotlib.pyplot as plt

delta = samples.diff().dropna()
delta["IPC"] = ...                # TODO: instructions per cycle in each interval
delta["L1_miss_per_kI"] = ...     # TODO: L1 data-cache misses per 1,000 instructions
time_ms = samples.time_ms.iloc[1:]

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(8, 5))
# TODO: plot IPC over time on ax1, and the misses on ax2; label the axes

✅

In [ ]:
from pandas.api.types import is_numeric_dtype

if not all(is_numeric_dtype(delta[c]) for c in ["IPC", "L1_miss_per_kI"]):
    print("✘ finish TODO 7 first: IPC and L1_miss_per_kI must be numbers")
else:
    high = delta.IPC.quantile(0.9)                 # the compute phases
    slow = delta.IPC < 0.5 * high                  # samples at less than half the compute IPC
    more_misses = delta.L1_miss_per_kI[slow].median() > delta.L1_miss_per_kI[~slow].median()
    print("✔ high-IPC compute phases, and slower phases with more cache misses"
          if slow.sum() >= 3 and more_misses
          else f"✘ IPC {delta.IPC.min():.2f} to {high:.2f}, {slow.sum()} slow samples: check your formulas")

❓ Find the four phases of `phases` in your plot. Right after the first compute phase, there's a short phase with only a few points and the lowest IPC. Which line of the code is it?

## 4. Wrap-up

Homework, the VM template, and the pinning reference solution: see the QR code on the slide.

Finished early, and on native Linux? `friday_extra.ipynb` watches the function from the kernel with eBPF (bpftrace): which files a cold start opens, and which system calls a request makes.

## 5. Cleaning up

The course left things on your laptop and in your cloud account. Remove them step by step, **after the homework** if you use your setup for it (keep the cloud function, the storage, and `results/` until then). Steps 1 and 2 need this notebook's kernel with section 1 and section 2's setup cells run; the others don't.

### Step 1: the cloud

▶️ Delete the cloud function SeBS created for you (`210.thumbnailer`; on AWS, its Function URL goes with it):

In [ ]:
print("deleted:", cloud.cleanup_functions(dry_run=False))

The rest is in your cloud's console. All of it has your laptop's prefix in its name (`phd` and six characters, e.g. `phd22d363`; the exact form varies between resources):

**AWS** (region `eu-central-1`, top right in the console):
1. **S3** → Buckets: `sebs-benchmarks-phd…`: select it, *Empty*, then *Delete*.
2. **CloudWatch** → Log groups: `/aws/lambda/sebs_phd…`: the logs of your function's invocations. Select, *Actions* → *Delete*.
3. **IAM** → Roles: `sebs-lambda-role`, the role your function ran with. Delete it only if nothing else in the account uses SeBS (an account we gave you: leave it).
4. **Lambda** → Functions: check that no `sebs_phd…` function is left.

**GCP** (project of your service account; region `europe-west1`):
1. **Cloud Storage** → Buckets: `sebs-benchmarks-phd…` and `sebs-deployment-phd…` (SeBS's), and `gcf-sources-…` (Google's copies of your function's code).
2. **Artifact Registry** → Repositories: `gcf-artifacts`, with the container images Google built from your function (e.g. `sebs__phd…__210__python__3__11__x64__gen1`). Delete the images, or the whole repository if you don't use Cloud Functions otherwise.
3. **Cloud Run functions** (Cloud Run → Services → "Go to Cloud Run functions gen1"): check that no `sebs_phd…` function is left.

A day later, look at your billing page: it should show nothing new.

### Step 2: the local functions and the storage

▶️ The local function containers (sections 2 and 3, and `friday_extra.ipynb`) and the stressors:

In [ ]:
st.cleanup_local(sebs_client, local)

▶️ The object storage (RustFS) container. Its data, the buckets with the benchmarks' inputs and outputs, stays in a Docker volume:

In [ ]:
st.stop_storage()

Then, in a terminal (from now on, everything is in a terminal, in the tutorial directory):

```bash
docker volume rm sebs-tutorial-rustfs     # the storage's data
docker ps -a                              # anything left from the course? spcleth/serverless-benchmarks or rustfs/rustfs images
docker rm -f <name>                       # ... remove it by its name
```

### Step 3: OpenWhisk (Thursday)

Thursday's last cell removed your OpenWhisk functions. The cluster itself is still there (stopped if you ran `openwhisk/stop.sh`):

```bash
./openwhisk/delete.sh        # answer y
```

It deletes the kind cluster `phdopen` (OpenWhisk with its database, all functions and their logs) and the local registry `phdopen-registry` with the function images in it. What stays:
- the registry's data: `docker volume ls` shows it as a volume with a long hexadecimal name and no other user; `docker volume prune` removes such unused volumes (all of them: check the list it shows first);
- the tools in `openwhisk/deps/` (removed with the directory, step 6);
- the `wsk` configuration in `~/.wskprops`: delete the file, unless you use `wsk` for another OpenWhisk.

### Step 5: Docker images

`docker images` lists them; the course's are:
- `spcleth/serverless-benchmarks:*`: SeBS's images, to build and run functions (also as `localhost:5001/spcleth/...`: the copies for OpenWhisk's registry);
- `kindest/node`: kind's Kubernetes nodes; `registry:2`: the local registry;
- `rustfs/rustfs`: the object storage;
- `sebs-tutorial/stress-ng`: the stressor of section 2.

Remove them with `docker rmi <image>` (only those you don't use otherwise), then `docker image prune` for the untagged layers left behind. `docker system df` shows how much space Docker uses.

### Step 6: files

Everything else is in the tutorial directory: SeBS and its Python environment (`serverless-benchmarks/`), the OpenWhisk tools (`openwhisk/deps/`), SeBS's cache (`cache/`), your measurements (`results/`, `outputs/`), and your cloud credentials (`configs/*credentials*.json`). Copy what you want to keep (your results, your homework), then delete the directory:

```bash
cd .. && rm -rf sebs-tutorial
```

uv keeps downloaded Python packages in its cache: `uv cache clean` removes them.